<a href="https://colab.research.google.com/github/TKhahahah/Data_Mining_FinalProject/blob/main/DTM_PJ1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Data

In [32]:
import pandas as pd
import os, sys
import numpy as np
import copy
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pickle

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
X_index = pd.read_csv('/content/drive/MyDrive/KKU3/X_variable(Index)_operation_v4.csv')
Y_rainfall = pd.read_csv('/content/drive/MyDrive/KKU3/Y_variable(Rainfall).csv')
# Convert 'DATE' column in Y_df to datetime objects if it's not already
X_index['DATE'] = pd.to_datetime(X_index['DATE'])

# Format the dates in Y_df as 'YYYY-MM-01'
X_index['DATE'] = X_index['DATE'].dt.strftime('%Y-%m-01')



node = pd.read_csv('/content/drive/MyDrive/KKU3/Node_table_TMD.csv')


path_feature = os.path.join("/content/drive/MyDrive/KKU3/lookup_table_reanalysis_v5.csv")
feature = pd.read_csv(path_feature, header= "infer")



In [8]:
X_index.head()

,DATE,MEIV2,RMM_AMPLITUDE,PHASE,BSISO1,BSISO1-Phase,DMI,ONI,PDO,SWM,NE
0,1981-01-01,0.35,1.79134,7.0,0.000,0.0,NaN,NaN,NaN,NaN,NaN
1,1981-02-01,0.23,2.04364,7.0,0.000,0.0,NaN,NaN,NaN,NaN,NaN
2,1981-03-01,0.33,3.16988,3.0,0.000,0.0,NaN,NaN,NaN,NaN,NaN
3,1981-04-01,0.42,3.22159,4.0,0.000,0.0,NaN,NaN,NaN,NaN,NaN
4,1981-05-01,0.24,0.00000,1.0,1.467,8.0,NaN,NaN,NaN,NaN,NaN


In [41]:
X_index.columns

Index(['DATE', 'MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase',
       'DMI', 'ONI', 'PDO', 'SWM', 'NE', 'MEIV2_lag1', 'MEIV2_lag2',
       'MEIV2_lag3', 'RMM_AMPLITUDE_lag1', 'PHASE_lag1', 'PDO_lag1',
       'PDO_lag2', 'ONI_lag1', 'ONI_lag2', 'DMI_lag1', 'DMI_lag2',
       'BSISO1_lag1', 'BSISO1-Phase_lag1', 'SWM_lag1'],
      dtype='object')

In [9]:
Y_rainfall.head()

,DATE,300201,300202,303201,303301,310201,327202,327301,327501,328201,...,566202,567201,568301,568401,568501,568502,570201,580201,581301,583201
0,1961-01-01,0.5,13.1,0.4,NaN,NaN,NaN,NaN,1.8,0.4,...,NaN,71.2,NaN,NaN,58.5,NaN,NaN,NaN,NaN,193.6
1,1961-02-01,1.1,4.4,5.7,NaN,NaN,NaN,NaN,33.3,2.4,...,NaN,89.9,NaN,NaN,13.2,NaN,NaN,NaN,NaN,76.7
2,1961-03-01,7.6,0.0,37.7,NaN,NaN,NaN,NaN,22.5,46.2,...,NaN,62.1,NaN,NaN,5.4,NaN,NaN,NaN,NaN,24.5
3,1961-04-01,94.6,27.1,118.8,NaN,NaN,NaN,NaN,65.2,58.6,...,NaN,124.1,NaN,NaN,154.9,NaN,NaN,NaN,NaN,42.3
4,1961-05-01,223.9,189.5,257.9,NaN,NaN,NaN,NaN,249.8,240.9,...,NaN,219.7,NaN,NaN,114.8,NaN,NaN,NaN,NaN,80.7


In [10]:
Y_rainfall.columns

Index(['DATE', '300201', '300202', '303201', '303301', '310201', '327202',
       '327301', '327501', '328201',
       ...
       '566202', '567201', '568301', '568401', '568501', '568502', '570201',
       '580201', '581301', '583201'],
      dtype='object', length=133)

In [11]:
node.head()

,Node,Lat,Long,cluster,quality,lag
0,300201,19.299814,97.972734,7.0,1,NaN
1,300202,18.176442,97.930608,7.0,1,NaN
2,303201,19.961219,99.881294,7.0,1,NaN
3,303301,19.871839,99.779203,7.0,1,NaN
4,310201,19.192933,99.883650,NaN,0,NaN


In [12]:
feature.head()

,Cluster,Node,Lat,Long,quality,lag
0,1,SWM,0.0,101.0,0,0
1,1,DMI,0.0,80.0,1,1
2,1,PDO,0.0,165.0,1,1
3,1,ONI,0.0,180.0,0,0
4,1,NE,10.0,110.0,1,0


# Cleansing

In [14]:
algo='LSTM_V23_11'

num_epoch = 50
window_size = 24  #Window size
horizon = 12   #Prediction horizon
TBPTT_K = 24

learn_rate =0.001
weight_decay =1e-4
cluster = 6 #choose cluster of the HII stations
k_fold_num=2
percentile_loss =0.95

# PARAMETERS FOR SMALL DATA
noise_level = 0.05  # Standard deviation of noise (5% of signal)

min_epochs = 20
patience = 100
LOG_EVERY = 20


hidden_size_list =[128]
num_layers_list =[2]
drop_out_list=[0.4]



In [15]:
Y_rainfall = Y_rainfall.rename(columns={'code': 'Node'})

Y_rainfall_clean = Y_rainfall.copy()
num_feature = len(feature[feature['Cluster']==cluster])

In [16]:

def apply_time_lag(df, column_name, lag):
    # Create a new column name with the lag suffix
    new_column_name = f"{column_name}_lag{lag}"

    # Shift the column values by the specified lag
    df[new_column_name] = df[column_name].shift(lag)

    return df

# Define the time lags for each climate index

time_lags = {

    'MEIV2': [1, 2, 3],
    'RMM_AMPLITUDE': [1],
    'PHASE': [1],
    'PDO': [1, 2],
    'ONI': [1, 2],
    'DMI': [1, 2],
    'BSISO1': [1],
    'BSISO1-Phase': [1],
    'SWM': [1],

}




# Apply time lags to each specified column in df_merge_type
for column, lag in time_lags.items():
      if column in X_index.columns:
            for lag_item in lag:
                X_index = apply_time_lag(X_index, column, lag_item)
      else:
        print(f"Warning: Column '{column}' not found in df_merge_type. Skipping.")


## start clean

In [19]:
HQ_date_get_month_df  =  Y_rainfall.copy()
Y_rainfall_clean =  Y_rainfall.copy()
HQ_date_get_month_df['DATE'] = pd.to_datetime(Y_rainfall['DATE']).dt.month

month_avg_list = []
for month in range(1, 12 + 1):
    # Filter the data for the current year
    HQ_data = HQ_date_get_month_df[HQ_date_get_month_df['DATE'] == month]

    # Average rainfall
    av_r = HQ_data.iloc[:, 1:].median()

    # Create a dictionary to hold the data for this year
    month_dict = {'MONTH': month}
    month_dict.update(av_r)

    # Append the data for this year to the list
    month_avg_list.append(month_dict)

HQ_month_avg = pd.DataFrame(month_avg_list)

for i in range(1, len(Y_rainfall_clean.columns)):
    for j in range(0, len(Y_rainfall_clean)):
        if(np.isnan(Y_rainfall_clean.iloc[j, i])):
             Y_rainfall_clean.iloc[j, i] = HQ_month_avg.iloc[pd.to_datetime(Y_rainfall_clean.iloc[:, 0]).dt.month[j]-1, i]

In [20]:
Y_rainfall_clean['DATE'].max()

'2025-03-01'

In [23]:
start_training_date = '1982-01-01'


X_index['DATE'] = pd.to_datetime(X_index['DATE'])
Y_rainfall_clean['DATE'] = pd.to_datetime(Y_rainfall_clean['DATE'])
max_date_X = X_index['DATE'].max()
max_date_Y = Y_rainfall_clean['DATE'].max()
# Minimum date possible

print("Maximum date possible : ",start_training_date)


# Maximum date possible
max_date_possible = min([max_date_X,max_date_Y])
print("Maximum date possible : ",max_date_possible)

#Make condition
con_date_X = (X_index['DATE'] >= start_training_date) & (X_index['DATE'] <= max_date_possible)
con_date_Y = (Y_rainfall_clean['DATE'] >= start_training_date) & (Y_rainfall_clean['DATE'] <= max_date_possible)

#Final select
X_index_interval_date = X_index.loc[con_date_X,:]
Y_rainfall_clean = Y_rainfall_clean.loc[con_date_Y,:]


Y_rainfall_interval_date = copy.deepcopy(Y_rainfall_clean)

Maximum date possible :  1982-01-01
Maximum date possible :  2025-03-01 00:00:00


In [24]:
start_training_date = Y_rainfall_clean['DATE'].min()

X_index['DATE'] = pd.to_datetime(X_index['DATE'])
Y_rainfall_clean['DATE'] = pd.to_datetime(Y_rainfall_clean['DATE'])
max_date_X = X_index['DATE'].max()
max_date_Y = Y_rainfall_clean['DATE'].max()
# Minimum date possible

print("Maximum date possible : ",start_training_date)


# Maximum date possible
max_date_possible = min([max_date_X,max_date_Y])
print("Maximum date possible : ",max_date_possible)

#Make condition
con_date_X = (X_index['DATE'] >= start_training_date) & (X_index['DATE'] <= max_date_possible)
con_date_Y = (Y_rainfall_clean['DATE'] >= start_training_date) & (Y_rainfall_clean['DATE'] <= max_date_possible)

#Final select
X_index_interval_date = X_index.loc[con_date_X,:]
Y_rainfall_clean = Y_rainfall_clean.loc[con_date_Y,:]


Y_rainfall_interval_date = copy.deepcopy(Y_rainfall_clean)

Maximum date possible :  1982-01-01 00:00:00
Maximum date possible :  2025-03-01 00:00:00


In [25]:
#Count null value of each indexs
print("Number of index columns which contain null value : ",len(X_index_interval_date.isnull().sum()[X_index_interval_date.isnull().sum()>0]), sep="")
print("Number of rainfall station which contain null value : ",len(Y_rainfall_interval_date.isnull().sum()[Y_rainfall_interval_date.isnull().sum()>0]), sep="")

Number of index columns which contain null value : 0
Number of rainfall station which contain null value : 0


In [26]:
def convert_columns_to_index_columns(df,col_name):
    df.loc[:,col_name] = pd.to_datetime(df[col_name])
    df = df.set_index(col_name, inplace=False)
    return df

In [27]:
X_index_df_ready = convert_columns_to_index_columns(X_index_interval_date, 'DATE')
Y_rainfall_df_ready = convert_columns_to_index_columns(Y_rainfall_interval_date, 'DATE')

In [28]:
print("nrow of X : {}".format(X_index_df_ready.shape[0]))
print("nrow of Y : {}".format(X_index_df_ready.shape[0]))

nrow of X : 519
nrow of Y : 519


In [31]:

#scaler_X = MinMaxScaler()
scaler_X = StandardScaler()

X_normalize_data = scaler_X.fit_transform(X_index_df_ready)
X_index_normalized_df = pd.DataFrame(X_normalize_data, columns=X_index_df_ready.columns)

scaler_Y = StandardScaler()

Y_normalize_data = scaler_Y.fit_transform(Y_rainfall_df_ready)
Y_rainfall_normalized_df = pd.DataFrame(Y_normalize_data, columns=Y_rainfall_df_ready.columns)

In [34]:
#Get date range from original files because after normalization index will reset
og_date_range = X_index_interval_date['DATE']
X_index_normalized_df.set_index(og_date_range, inplace=True)
Y_rainfall_normalized_df.set_index(og_date_range, inplace=True)

n_feature_climate = X_index_normalized_df.shape[1]
feature_climate = X_index_normalized_df.columns
print("Number of climate index feature : ",n_feature_climate)
print("All climate index : \n",feature_climate)

Number of climate index feature :  24
All climate index : 
 Index(['MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase', 'DMI',
       'ONI', 'PDO', 'SWM', 'NE', 'MEIV2_lag1', 'MEIV2_lag2', 'MEIV2_lag3',
       'RMM_AMPLITUDE_lag1', 'PHASE_lag1', 'PDO_lag1', 'PDO_lag2', 'ONI_lag1',
       'ONI_lag2', 'DMI_lag1', 'DMI_lag2', 'BSISO1_lag1', 'BSISO1-Phase_lag1',
       'SWM_lag1'],
      dtype='object')


In [35]:
X_index_normalized_df['Month_sin'] = np.sin(2 * np.pi * X_index_normalized_df.index.month / 12)
X_index_normalized_df['Month_cos'] = np.cos(2 * np.pi * X_index_normalized_df.index.month / 12)

In [36]:
station_id_list = Y_rainfall_normalized_df.columns
list_table = []
for code_st in station_id_list:
  #print(code_st)

  #print(static_value)
  eindices_df = X_index_normalized_df.copy()



  eindices_df[code_st] = Y_rainfall_normalized_df[code_st]

  list_table.append(eindices_df)

In [37]:
n_feature_climate = X_index_normalized_df.shape[1]
feature_climate = X_index_normalized_df.columns
print("Number of climate index feature : ",n_feature_climate)
print("All climate index : \n",feature_climate)

Number of climate index feature :  26
All climate index : 
 Index(['MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase', 'DMI',
       'ONI', 'PDO', 'SWM', 'NE', 'MEIV2_lag1', 'MEIV2_lag2', 'MEIV2_lag3',
       'RMM_AMPLITUDE_lag1', 'PHASE_lag1', 'PDO_lag1', 'PDO_lag2', 'ONI_lag1',
       'ONI_lag2', 'DMI_lag1', 'DMI_lag2', 'BSISO1_lag1', 'BSISO1-Phase_lag1',
       'SWM_lag1', 'Month_sin', 'Month_cos'],
      dtype='object')


In [38]:
n_feature_climate

26

In [39]:
feature_climate

Index(['MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase', 'DMI',
       'ONI', 'PDO', 'SWM', 'NE', 'MEIV2_lag1', 'MEIV2_lag2', 'MEIV2_lag3',
       'RMM_AMPLITUDE_lag1', 'PHASE_lag1', 'PDO_lag1', 'PDO_lag2', 'ONI_lag1',
       'ONI_lag2', 'DMI_lag1', 'DMI_lag2', 'BSISO1_lag1', 'BSISO1-Phase_lag1',
       'SWM_lag1', 'Month_sin', 'Month_cos'],
      dtype='object')

In [40]:
X_index_normalized_df

,MEIV2,RMM_AMPLITUDE,PHASE,BSISO1,BSISO1-Phase,DMI,ONI,PDO,SWM,NE,...,PDO_lag2,ONI_lag1,ONI_lag2,DMI_lag1,DMI_lag2,BSISO1_lag1,BSISO1-Phase_lag1,SWM_lag1,Month_sin,Month_cos
DATE,,,,,,,,,,,,,,,,,,,,,
1982-01-01,-0.187714,0.714459,-1.386540,-0.948166,-0.832762,-0.023677,-0.140501,0.366424,1.469381,1.793781,...,0.748840,-0.324733,-0.452551,-0.523783,-0.532089,-0.948166,-0.832762,1.136801,5.000000e-01,8.660254e-01
1982-02-01,-0.102040,0.281839,1.174365,-0.948166,-0.832762,1.265857,-0.042981,0.316895,0.837625,1.758152,...,0.634039,-0.140014,-0.324464,-0.021566,-0.522618,-0.948166,-0.832762,1.466406,8.660254e-01,5.000000e-01
1982-03-01,-1.056691,0.443087,0.747548,-0.948166,-0.832762,0.981589,-0.074650,0.177455,0.424145,1.158306,...,0.361165,-0.042503,-0.139761,1.268030,-0.020539,-0.948166,-0.832762,0.835332,1.000000e+00,6.123234e-17
1982-04-01,-0.261148,0.738476,1.601183,-0.948166,-0.832762,1.140032,0.200470,-0.098664,0.264021,1.416414,...,0.311641,-0.074169,-0.042259,0.983748,1.268705,-0.948166,-0.832762,0.422298,8.660254e-01,-5.000000e-01
1982-05-01,-0.579365,-0.930963,-0.532905,0.774939,-0.170381,1.531242,0.516375,-0.382280,-0.468001,0.380173,...,0.172211,0.200925,-0.073923,1.142199,0.984501,-0.948166,-0.832762,0.262348,5.000000e-01,-8.660254e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-01,0.118264,1.033203,-0.106088,-0.948166,-0.832762,-1.326749,-0.358461,-1.858967,-0.317086,0.487085,...,-1.454253,-0.210207,-0.189716,-1.170573,0.467312,1.437983,1.485571,-1.284616,-5.000000e-01,8.660254e-01
2024-12-01,0.399764,1.280297,0.320730,-0.948166,-0.832762,-1.191502,-0.533606,-0.917404,0.952684,0.793703,...,-1.731047,-0.357952,-0.209948,-1.324702,-1.169232,-0.948166,-0.832762,-0.318132,-2.449294e-16,1.000000e+00
2025-01-01,0.509916,1.073795,-0.532905,-0.948166,-0.832762,-1.005699,-0.629968,-0.503890,1.974034,1.132187,...,-1.864045,-0.533081,-0.357680,-1.189448,-1.323318,-0.948166,-0.832762,0.950266,5.000000e-01,8.660254e-01


In [42]:
Y_normalize_data

array([[-1.00118725, -0.98116192, -0.95348275, ..., -0.95299072,
        -1.14774134, -0.87020789],
       [-1.0195053 , -0.98425392, -0.99367516, ..., -0.9326291 ,
        -1.1459257 , -0.83825998],
       [-1.0195053 , -0.98425392, -0.9781623 , ..., -0.94580427,
        -0.81003255, -0.88249555],
       ...,
       [-1.0195053 , -0.98425392, -0.99367516, ..., -0.26788208,
        -0.08256666,  0.78985493],
       [-1.01275655, -0.57817157, -0.77931565, ..., -0.52779218,
        -0.39969821, -0.7362724 ],
       [-0.98094098, -0.98425392, -0.78354643, ..., -0.64277544,
        -0.59276112, -0.8337545 ]])